In [11]:
"""
Pipeline de Data Scraping pour Tenymalagasy.org
Version simplifiée - Export direct en CSV et JSON (sans base de données)
"""

import requests
from bs4 import BeautifulSoup
import json
import csv
import time
from typing import Dict, List, Optional
from urllib.parse import urljoin
import logging

# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('scraping.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)

class TenyMalagasyPipeline:
    """Pipeline simplifié pour le scraping de tenymalagasy.org"""
    
    def __init__(self, base_url: str = "https://tenymalagasy.org", delay: float = 1.5):
        self.base_url = base_url
        self.delay = delay
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Educational Research Bot)',
            'Accept-Language': 'mg,fr,en'
        })
        self.all_words = []  # Stockage en mémoire
        self.alpha_ranges = []  # Liste des plages alphabétiques
    
    def fetch_page(self, url: str) -> Optional[BeautifulSoup]:
        """Récupère et parse une page"""
        try:
            time.sleep(self.delay)
            full_url = urljoin(self.base_url, url) if not url.startswith('http') else url
            response = self.session.get(full_url, verify=False)
            response.raise_for_status()
            response.encoding = 'utf-8'
            return BeautifulSoup(response.text, 'html.parser')
        except requests.RequestException as e:
            logging.error(f"Erreur lors de la récupération de {url}: {e}")
            return None
    
    def extract_homepage_sections(self) -> List[Dict]:
        """Extrait toutes les sections depuis la page d'accueil"""
        logging.info("Extraction des sections de la page d'accueil...")
        soup = self.fetch_page('/bins/homePage')
        
        if not soup:
            return []
        
        sections = []
        nav_links = soup.find_all('a', href=True)
        
        for link in nav_links:
            href = link.get('href', '')
            text = link.get_text(strip=True)
            
            if text and href and '/bins/' in href:
                sections.append({
                    'name': text,
                    'url': href
                })
        
        logging.info(f"✓ {len(sections)} sections trouvées")
        return sections
    
    def extract_alpha_ranges(self, section_url: str = '/bins/alphaLists') -> List[Dict]:
        """Extrait les plages alphabétiques (ex: -a - abe, abede - akakokako)"""
        logging.info(f"Extraction des plages alphabétiques...")
        soup = self.fetch_page(section_url)
        
        if not soup:
            return []
        
        alpha_ranges = []
        links = soup.find_all('a', href=True)
        
        for link in links:
            text = link.get_text(strip=True)
            href = link.get('href', '')
            
            # Les plages alphabétiques ont un format comme "-a - abe", "abede - akakokako"
            if '-' in text and href and len(text) < 50:
                alpha_ranges.append({
                    'range': text,
                    'url': href
                })
        
        self.alpha_ranges = alpha_ranges
        logging.info(f"✓ {len(alpha_ranges)} plages alphabétiques trouvées")
        return alpha_ranges
    
    def extract_words_from_range(self, range_url: str, range_name: str = '') -> List[Dict]:
        """Extrait tous les mots d'une plage alphabétique"""
        soup = self.fetch_page(range_url)
        
        if not soup:
            return []
        
        words = []
        
        # Chercher la structure du tableau
        # Colonne 1: Teny malagasy, Colonne 2: Anglisy, Colonne 3: Frantsay
        rows = soup.find_all('tr')
        
        for row in rows:
            cols = row.find_all('td')
            
            if len(cols) >= 3:
                malagasy_col = cols[0]
                english_col = cols[1]
                french_col = cols[2]
                
                # Extraire le mot malgache et son lien
                word_link = malagasy_col.find('a', href=True)
                if word_link:
                    word_data = {
                        'malagasy': word_link.get_text(strip=True),
                        'english': english_col.get_text(strip=True),
                        'french': french_col.get_text(strip=True),
                        'alpha_range': range_name,
                        'url': self.base_url + word_link.get('href', '')
                    }
                    words.append(word_data)
        
        logging.info(f"  ✓ {len(words)} mots extraits de '{range_name}'")
        return words
    
    def scrape_complete_dictionary(self, max_ranges: int = None):
        """Scrape complet du dictionnaire"""
        start_time = time.time()
        logging.info("\n=== DÉBUT DU SCRAPING ===\n")
        
        # Extraire les plages alphabétiques
        alpha_ranges = self.extract_alpha_ranges()
        
        if max_ranges:
            alpha_ranges = alpha_ranges[:max_ranges]
            logging.info(f"Limitation à {max_ranges} plages pour ce test\n")
        
        # Pour chaque plage, extraire tous les mots
        for i, ar in enumerate(alpha_ranges, 1):
            logging.info(f"[{i}/{len(alpha_ranges)}] Plage: {ar['range']}")
            
            words = self.extract_words_from_range(ar['url'], ar['range'])
            self.all_words.extend(words)
            
            # Pause tous les 5 plages
            if i % 5 == 0:
                logging.info("  ⏸ Pause de 10 secondes...")
                time.sleep(10)
        
        duration = time.time() - start_time
        logging.info(f"\n=== SCRAPING TERMINÉ ===")
        logging.info(f"Total de mots collectés: {len(self.all_words)}")
        logging.info(f"Durée totale: {duration:.2f} secondes")
        
        return len(self.all_words)
    
    def scrape_specific_ranges(self, range_patterns: List[str]):
        """Scrape seulement certaines plages alphabétiques"""
        if not self.alpha_ranges:
            self.extract_alpha_ranges()
        
        for pattern in range_patterns:
            matching = [r for r in self.alpha_ranges if pattern.lower() in r['range'].lower()]
            
            for ar in matching:
                logging.info(f"Scraping de la plage: {ar['range']}")
                words = self.extract_words_from_range(ar['url'], ar['range'])
                self.all_words.extend(words)
    
    def export_to_json(self, output_file: str = "malagasy_dictionary.json"):
        """Exporte toutes les données en JSON"""
        if not self.all_words:
            logging.warning("Aucune donnée à exporter!")
            return 0
        
        # Trier par ordre alphabétique malgache
        sorted_words = sorted(self.all_words, key=lambda x: x['malagasy'])
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(sorted_words, f, ensure_ascii=False, indent=2)
        
        logging.info(f"✓ {len(sorted_words)} mots exportés vers {output_file}")
        return len(sorted_words)
    
    def export_to_csv(self, output_file: str = "malagasy_dictionary.csv"):
        """Exporte toutes les données en CSV"""
        if not self.all_words:
            logging.warning("Aucune donnée à exporter!")
            return 0
        
        # Trier par ordre alphabétique malgache
        sorted_words = sorted(self.all_words, key=lambda x: x['malagasy'])
        
        with open(output_file, 'w', newline='', encoding='utf-8') as f:
            if sorted_words:
                fieldnames = ['malagasy', 'english', 'french', 'alpha_range', 'url']
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(sorted_words)
        
        logging.info(f"✓ {len(sorted_words)} mots exportés vers {output_file}")
        return len(sorted_words)
    
    def get_statistics(self) -> Dict:
        """Statistiques des données collectées"""
        if not self.all_words:
            return {'total_words': 0, 'alpha_ranges_covered': 0, 'top_ranges': []}
        
        # Compter les mots par plage alphabétique
        range_counts = {}
        for word in self.all_words:
            range_name = word.get('alpha_range', 'Unknown')
            range_counts[range_name] = range_counts.get(range_name, 0) + 1
        
        # Top 5 des plages
        top_ranges = sorted(range_counts.items(), key=lambda x: x[1], reverse=True)[:5]
        
        return {
            'total_words': len(self.all_words),
            'alpha_ranges_covered': len(range_counts),
            'top_ranges': top_ranges,
            'all_range_counts': range_counts
        }
    
    def search_word(self, query: str) -> List[Dict]:
        """Recherche un mot dans les données collectées"""
        query_lower = query.lower()
        results = []
        
        for word in self.all_words:
            if (query_lower in word['malagasy'].lower() or 
                query_lower in word['english'].lower() or 
                query_lower in word['french'].lower()):
                results.append(word)
        
        return results[:20]  # Limiter à 20 résultats
    
    def get_words_by_range(self, range_name: str) -> List[Dict]:
        """Récupère tous les mots d'une plage spécifique"""
        return [w for w in self.all_words if w['alpha_range'] == range_name]
    
    def clear_data(self):
        """Efface toutes les données en mémoire"""
        self.all_words = []
        self.alpha_ranges = []
        logging.info("✓ Données effacées de la mémoire")


# ========== EXEMPLES D'UTILISATION ==========

def example_usage():
    """Exemples d'utilisation du pipeline"""
    
    pipeline = TenyMalagasyPipeline(delay=1.5)
    
    print("\n" + "="*60)
    print("   PIPELINE DE SCRAPING TENYMALAGASY.ORG")
    print("="*60 + "\n")
    
    # OPTION 1: Test avec quelques plages (RECOMMANDÉ pour débuter)
    print("🔍 Mode: Test avec 3 plages alphabétiques\n")
    pipeline.scrape_complete_dictionary()
    
    # OPTION 2: Scraper des plages spécifiques
    # print("🔍 Mode: Plages spécifiques\n")
    # pipeline.scrape_specific_ranges(['a - abe', 'ba - broety'])
    
    # OPTION 3: Scraper TOUT le dictionnaire (ATTENTION: peut prendre des heures!)
    # print("🔍 Mode: Dictionnaire complet\n")
    # pipeline.scrape_complete_dictionary()
    
    # Export des données
    print("\n" + "-"*60)
    print("📤 EXPORT DES DONNÉES")
    print("-"*60)
    pipeline.export_to_json("malagasy_dictionary.json")
    pipeline.export_to_csv("malagasy_dictionary.csv")
    
    # Statistiques
    print("\n" + "-"*60)
    print("📊 STATISTIQUES")
    print("-"*60)
    stats = pipeline.get_statistics()
    print(f"Total de mots collectés: {stats['total_words']}")
    print(f"Plages alphabétiques: {stats['alpha_ranges_covered']}")
    print(f"\nTop 5 des plages les plus riches:")
    for i, (range_name, count) in enumerate(stats['top_ranges'], 1):
        print(f"  {i}. {range_name}: {count} mots")
    
    # Exemple de recherche
    print("\n" + "-"*60)
    print("🔎 EXEMPLE DE RECHERCHE")
    print("-"*60)
    search_term = 'teny'
    results = pipeline.search_word(search_term)
    print(f"Recherche pour '{search_term}': {len(results)} résultats\n")
    
    for i, r in enumerate(results[:5], 1):
        print(f"{i}. {r['malagasy']}")
        print(f"   EN: {r['english']}")
        print(f"   FR: {r['french']}")
        print(f"   Plage: {r['alpha_range']}\n")
    
    print("="*60)
    print("✅ PROCESSUS TERMINÉ")
    print("="*60 + "\n")


if __name__ == "__main__":
    example_usage()

2025-12-18 08:55:33,260 - INFO - 
=== DÉBUT DU SCRAPING ===

2025-12-18 08:55:33,261 - INFO - Extraction des plages alphabétiques...



   PIPELINE DE SCRAPING TENYMALAGASY.ORG

🔍 Mode: Test avec 3 plages alphabétiques



C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1045: InsecureRequestWarning: Unverified HTTPS request is being made to host 'tenymalagasy.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
2025-12-18 08:55:36,312 - INFO - ✓ 63 plages alphabétiques trouvées
2025-12-18 08:55:36,312 - INFO - [1/63] Plage: Fitenim-paritra
C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1045: InsecureRequestWarning: Unverified HTTPS request is being made to host 'tenymalagasy.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
2025-12-18 08:55:38,177 - INFO -   ✓ 2 mots extraits de 'Fitenim-paritra'
2025-12-18 08:55:38,178 - INFO - [2/63] Plage: Anaran-tsamirery
C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1045: InsecureRequestWarning: Un


------------------------------------------------------------
📤 EXPORT DES DONNÉES
------------------------------------------------------------


2025-12-18 09:01:26,361 - INFO - ✓ 130350 mots exportés vers malagasy_dictionary.json
2025-12-18 09:01:26,972 - INFO - ✓ 130350 mots exportés vers malagasy_dictionary.csv



------------------------------------------------------------
📊 STATISTIQUES
------------------------------------------------------------
Total de mots collectés: 130350
Plages alphabétiques: 17

Top 5 des plages les plus riches:
  1. -: 111049 mots
  2. h-: 4231 mots
  3. FX-Mahah: 3385 mots
  4. -ay: 3216 mots
  5. fao-: 2404 mots

------------------------------------------------------------
🔎 EXEMPLE DE RECHERCHE
------------------------------------------------------------
Recherche pour 'teny': 20 résultats

1. Teny
   EN: 
   FR: Teny
   Plage: Fitenim-paritra

2. Teny
   EN: 
   FR: Teny
   Plage: Anaran-tsamirery

3. Teny
   EN: 
   FR: Teny
   Plage: Fitsipi-pitenenana

4. Teny
   EN: 
   FR: Teny
   Plage: Mpiara-miasa

5. Teny
   EN: 
   FR: Teny
   Plage: Sokajin-teny

✅ PROCESSUS TERMINÉ

